# Notebook Workflow Helpers for DGN4AVBP

This notebook mirrors the helper utilities that were previously exposed as a Python module.
It keeps the training-time architecture defined in `train_dgnavbp.py` while offering
notebook-friendly utilities for inspecting batches, constructing the diffusion components,
and plotting quick predictions.


In [ ]:
from __future__ import annotations

import logging
import os
from dataclasses import dataclass
from typing import List, Optional, Sequence, Tuple, Type

import lightning as L
import torch
from lightning.pytorch.callbacks import Callback, ModelCheckpoint, RichProgressBar
from lightning.pytorch.callbacks.progress.rich_progress import RichProgressBarTheme
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger
from torch_geometric.data import Batch, Data
from torchvision import transforms as T

from Dataset import create_cfd_datamodule
from utils import create_data_list, create_graph_data, load_coo_data, read_metadata
from dgn4avbp.callbacks import StepTimeTracker
from dgn4avbp.diffusion_process import DiffusionProcess
from dgn4avbp.dgn_model import DiffusionGraphNet
from dgn4avbp.lit_dgn import LitDiffusionCFD
from dgn4avbp.losses import HybridLoss
from dgn4avbp.step_sampler import ImportanceStepSampler
from dgn4avbp.transform_locals import (
    EnsureEdgeAttrFromPos,
    MeshCoarsening,
    ScaleEdgeAttr,
    ZScoreTarget,
)

log = logging.getLogger(__name__)
if not logging.getLogger().handlers:
    logging.basicConfig(level=logging.INFO)


## Combined Loader Utilities

Normalize batches coming from `CombinedLoader` and provide quick inspection helpers, mirroring the
functions used throughout the notebook workflow and `train_dgnavbp.py`.


In [ ]:
def get_sequence_from_combined(item: object) -> List[Batch]:
    """Normalize the nested structure returned by ``CombinedLoader``."""

    if isinstance(item, tuple) and len(item) == 3:
        batches, _batch_idx, _dl_idx = item
    else:
        batches = item

    if isinstance(batches, dict):
        sequence = next(iter(batches.values()))
    else:
        sequence = batches

    if isinstance(sequence, tuple):
        sequence = list(sequence)
    return list(sequence)


def print_batch_info(sequence: Sequence[Batch]) -> None:
    """Pretty-print a temporal batch for quick inspection."""

    log.info("Number of time steps in batch: %s", len(sequence))
    for t, g in enumerate(sequence):
        if isinstance(g, dict) and "main" in g:
            g = g["main"]
        if not isinstance(g, (Batch, Data)):
            log.info("  Time step %s: unexpected type %s", t + 1, type(g))
            continue
        log.info("  Time step %s:", t + 1)
        log.info("    Number of graphs in batch: %s", g.num_graphs)
        log.info("    Total number of nodes: %s", g.num_nodes)
        log.info("    Total number of edges: %s", g.num_edges)
        if hasattr(g, "target"):
            log.info("    target shape: %s", tuple(g.target.shape))
        if hasattr(g, "edge_attr"):
            log.info("    edge_attr shape: %s", tuple(g.edge_attr.shape))
        if hasattr(g, "cells") and isinstance(g.cells, list) and g.cells and g.cells[0] is not None:
            log.info("    Total number of cells: %s", len(g.cells[0]))
        if hasattr(g, "time"):
            log.info(
                "    time mean per-graph: %s",
                g.time.view(g.num_graphs, -1).mean(dim=-1),
            )


## Dataset Helpers

Create the `LightningDataModule` with the same preprocessing steps used for training, including
Z-score normalization and mesh coarsening.


In [ ]:
def _sample_graphs_for_stats(meta: dict, k: int = 200):
    pos0, edge_index0, cells0 = load_coo_data(
        meta["coo_file"], meta["mesh_file"], meta["coordinate_paths"]
    )
    file_list, it_list_total = create_data_list(
        [meta["solution_directory"]],
        meta["seq_len"],
        meta["solution_prefix"],
    )
    k = min(k, len(it_list_total))
    samples = []
    for idx in range(k):
        case, _sim, t = it_list_total[idx][0]
        fpath = file_list[str(case)][str(t)]
        g = create_graph_data(pos0, edge_index0, fpath, meta, cells0)
        samples.append(g)
    return samples, pos0, edge_index0


def _estimate_mesh_scales(pos: torch.Tensor, edge_index: torch.Tensor, quantile: float = 0.5):
    row, col = edge_index
    edge_len = (pos[col] - pos[row]).norm(dim=1)
    h = edge_len.quantile(quantile).item()
    rel_pos_scaling = [h, 2 * h, 4 * h, 8 * h]
    return h, rel_pos_scaling


class NotebookCFDDataModule(L.LightningDataModule):
    """``LightningDataModule`` carrying the preprocessing used for training."""

    def __init__(self, metadata_files: Sequence[str], train_val_split: float = 0.8):
        super().__init__()
        self.metadata_files = list(metadata_files)
        metadata = [read_metadata(mf) for mf in metadata_files]
        self.batch_sizes = [info["batch_size"] for info in metadata]
        self.loader_types = ["default"] * len(metadata_files)
        self.start_idx = [0] * len(metadata_files)
        self.split = train_val_split

        self.graph_transform = None
        self._zscore_mean = None
        self._zscore_std = None
        self._rel_pos_scales = None
        self._h = None

    def setup(self, stage: str) -> None:
        meta0 = read_metadata(self.metadata_files[0])
        samples, pos0, edge_index0 = _sample_graphs_for_stats(meta0, k=200)

        tgt = torch.cat([g.target for g in samples], dim=0)
        self._zscore_mean = tgt.mean(0)
        self._zscore_std = tgt.std(0)
        log.info("zscore_mean=%s zscore_std=%s", self._zscore_mean.tolist(), self._zscore_std.tolist())

        self._h, self._rel_pos_scales = _estimate_mesh_scales(pos0, edge_index0)
        log.info("median_h=%.6e rel_pos_scales=%s", self._h, self._rel_pos_scales)

        self.graph_transform = T.Compose(
            [
                EnsureEdgeAttrFromPos(),
                ScaleEdgeAttr(1 / self._h),
                ZScoreTarget(self._zscore_mean, self._zscore_std),
                MeshCoarsening(
                    num_scales=5,
                    max_indegree=None,
                    rel_pos_scaling=None,
                    scalar_rel_pos=False,
                ),
            ]
        )

        train_loader_kwargs = self._loader_kwargs(num_workers=1, prefetch_factor=2)
        val_loader_kwargs = self._loader_kwargs(num_workers=1, prefetch_factor=2)
        log.info("Train DataLoader kwargs: %s", train_loader_kwargs)
        log.info("Val DataLoader kwargs: %s", val_loader_kwargs)

        if stage == "fit":
            self.train_cfd_datamodule = create_cfd_datamodule(
                self.metadata_files,
                self.batch_sizes,
                self.loader_types,
                self.start_idx,
                shuffle=True,
                split=self.split,
                flag="train",
                collater_transform=self.graph_transform,
                dataloader_kwargs=train_loader_kwargs,
            )
            self.val_cfd_datamodule = create_cfd_datamodule(
                self.metadata_files,
                self.batch_sizes,
                self.loader_types,
                self.start_idx,
                shuffle=False,
                split=self.split,
                flag="val",
                collater_transform=self.graph_transform,
                dataloader_kwargs=val_loader_kwargs,
                cache_sequences=True,
            )
        if stage in {"test", "predict"}:
            self.val_cfd_datamodule = create_cfd_datamodule(
                self.metadata_files,
                self.batch_sizes,
                self.loader_types,
                self.start_idx,
                shuffle=False,
                split=self.split,
                flag="val",
                collater_transform=self.graph_transform,
                dataloader_kwargs=val_loader_kwargs,
                cache_sequences=True,
            )

    def _loader_kwargs(self, num_workers: int, *, prefetch_factor: int) -> dict:
        kwargs = {
            "num_workers": num_workers,
            "prefetch_factor": prefetch_factor,
        }
        if num_workers > 0:
            kwargs["persistent_workers"] = True
        if torch.cuda.is_available():
            kwargs["pin_memory"] = True
        return kwargs

    def train_dataloader(self):
        return self.train_cfd_datamodule.get_combined_loader()

    def val_dataloader(self):
        return self.val_cfd_datamodule.get_combined_loader(mode="sequential")

    def test_dataloader(self):
        return self.val_cfd_datamodule.get_combined_loader(mode="sequential")

    def predict_dataloader(self):
        return self.val_cfd_datamodule.get_combined_loader(mode="sequential")


## Model and Trainer Builders

Factories that mirror the training script configuration, so the notebook can instantiate the
same diffusion model, Lightning module, and trainer as `train_dgnavbp.py`.


In [ ]:
DEFAULT_METADATA_FILES = [
    os.environ.get("DGN4AVBP_METADATA", "/scratch/coop/theret/HIT_LES_FORCED/metadata.yaml")
]


ARCHITECTURE = {
    "in_node_features": 3,
    "cond_node_features": 0,
    "cond_edge_features": 3,
    "depths": [3, 3, 3, 3, 3],
    "fnns_width": 128,
    "aggr": "sum",
    "dropout": 0.1,
    "dim": 3,
    "scalar_rel_pos": False,
}


@dataclass
class NotebookComponents:
    datamodule: NotebookCFDDataModule
    diffusion_process: DiffusionProcess
    net: DiffusionGraphNet
    lit_module: LitDiffusionCFD
    step_sampler_factory: Type[ImportanceStepSampler]


def build_components(
    metadata_files: Sequence[str],
    *,
    train_val_split: float = 0.8,
    lr: float = 1e-4,
) -> NotebookComponents:
    """Instantiate the diffusion model and datamodule using the training architecture."""

    diffusion_process = DiffusionProcess(num_steps=1000, schedule_type="linear")
    net = DiffusionGraphNet(
        diffusion_process=diffusion_process,
        learnable_variance=True,
        arch=dict(ARCHITECTURE),
    )

    criterion = HybridLoss()
    step_sampler_factory = ImportanceStepSampler

    lit_module = LitDiffusionCFD(
        net=net,
        diffusion_process=diffusion_process,
        criterion=criterion,
        step_sampler_factory=step_sampler_factory,
        lr=lr,
        scheduler_cfg={"factor": 0.1, "patience": 250},
        pack_mode=None,
        pack_win_len=10,
        pack_stride=1,
        pack_select="random",
        y_idx=[0, 1, 2],
        cond_idx=None,
    )

    dm = NotebookCFDDataModule(metadata_files, train_val_split=train_val_split)

    return NotebookComponents(
        datamodule=dm,
        diffusion_process=diffusion_process,
        net=net,
        lit_module=lit_module,
        step_sampler_factory=step_sampler_factory,
    )


def build_progress_bar() -> Optional[RichProgressBar]:
    if not os.isatty(1):
        return None
    if os.environ.get("LIGHTNING_DISABLE_PROGRESS_BAR", "0") == "1":
        return None
    return RichProgressBar(
        theme=RichProgressBarTheme(
            description="green_yellow",
            progress_bar="green1",
            progress_bar_finished="green1",
            progress_bar_pulse="#6206E0",
            batch_progress="green_yellow",
            time="grey82",
            processing_speed="grey82",
            metrics="grey82",
            metrics_text_delimiter="
",
            metrics_format=".3e",
        )
    )


def build_trainer(
    *,
    log_dir: Optional[str] = None,
    max_epochs: int = 5000,
    limit_train_batches: Optional[int] = 16,
    limit_val_batches: Optional[int] = 4,
    accumulate_grad_batches: int = 4,
    gradient_clip_val: float = 1.0,
) -> Tuple[L.Trainer, List[Callback]]:
    """Create a trainer mirroring the defaults from ``train_dgnavbp.py``."""

    log_dir = log_dir or os.environ.get(
        "LOG_DIR", f"/scratch/{os.environ.get('USER', 'user')}/logs"
    )
    os.makedirs(log_dir, exist_ok=True)

    ckpt_dir = os.path.join(log_dir, "checkpoints")
    os.makedirs(ckpt_dir, exist_ok=True)

    checkpoint_cb = ModelCheckpoint(
        dirpath=ckpt_dir,
        filename="notebook-{epoch}",
        monitor="val/loss",
        mode="min",
        save_top_k=3,
        save_last=True,
    )

    progress_bar = build_progress_bar()
    callbacks: List[Callback] = [
        checkpoint_cb,
        StepTimeTracker(warmup_batches=0, log_every_n_steps=1),
    ]
    if progress_bar is not None:
        callbacks.insert(0, progress_bar)

    trainer = L.Trainer(
        max_epochs=max_epochs,
        accelerator="auto",
        precision="16-mixed",
        callbacks=callbacks,
        logger=[
            CSVLogger(save_dir=log_dir, name="lightning_csv"),
            TensorBoardLogger(save_dir=log_dir, name="lightning_tb"),
        ],
        enable_progress_bar=progress_bar is not None,
        log_every_n_steps=1,
        limit_val_batches=limit_val_batches,
        limit_train_batches=limit_train_batches,
        accumulate_grad_batches=accumulate_grad_batches,
        gradient_clip_val=gradient_clip_val,
        default_root_dir=log_dir,
    )
    return trainer, callbacks


## Quick Visualization Helper

Plot predictions emitted by `LitDiffusionCFD.predict` for rapid inspection inside the notebook.


In [ ]:
def quick_plot_pred(out: dict, dm: NotebookCFDDataModule, graph_id: int = 0, comp: str = "mag", quiver: bool = True, save: Optional[str] = None):
    """Plot predictions returned by ``LitDiffusionCFD.predict``."""

    import numpy as np
    import matplotlib.pyplot as plt

    pred_norm = out["pred_norm"]
    pos = out["pos"]
    batch_idx = out["batch"]

    if isinstance(pred_norm, torch.Tensor):
        pred_norm = pred_norm.detach().cpu().numpy()
    if isinstance(pos, torch.Tensor):
        pos = pos.detach().cpu().numpy()
    if isinstance(batch_idx, torch.Tensor):
        batch_idx = batch_idx.detach().cpu().numpy()

    mean = dm._zscore_mean.cpu().numpy()
    std = dm._zscore_std.cpu().numpy()
    pred = pred_norm * std + mean

    mask = batch_idx == graph_id
    xy = pos[mask, :2]
    x, y = xy[:, 0], xy[:, 1]

    if comp in ("u", 0):
        z = pred[mask, 0]
    elif comp in ("v", 1):
        z = pred[mask, 1]
    elif comp in ("w", 2):
        z = pred[mask, 2]
    else:
        z = np.linalg.norm(pred[mask, :3], axis=1)

    plt.figure(figsize=(6, 5), dpi=120)
    sc = plt.scatter(x, y, c=z, s=8)
    plt.colorbar(sc, label=str(comp))
    plt.gca().set_aspect("equal", "box")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"Prediction: {comp}")

    if quiver:
        u_comp = pred[mask, 0]
        v_comp = pred[mask, 1]
        step = max(len(x) // 150, 1)
        plt.quiver(
            x[::step],
            y[::step],
            u_comp[::step],
            v_comp[::step],
            angles="xy",
            scale_units="xy",
            scale=None,
            width=0.002,
            alpha=0.8,
        )

    plt.tight_layout()
    if save:
        plt.savefig(save, bbox_inches="tight")
    plt.show()


## Exported Helpers

Keep track of the most common entry points so that the notebook can still behave like a module
when imported via `jupyter nbconvert --to python` or similar workflows.


In [ ]:
__all__ = [
    "DEFAULT_METADATA_FILES",
    "ARCHITECTURE",
    "NotebookCFDDataModule",
    "NotebookComponents",
    "build_components",
    "build_trainer",
    "get_sequence_from_combined",
    "print_batch_info",
    "quick_plot_pred",
]
